# CNU Campus ChatBot — Colab 실행 노트북
**박연진 | 자연어처리 텀프로젝트**

## 실행 순서
```
[최초 1회]  셀 A → 런타임 재시작 → 셀 B → 셀 0 이후 순서대로
[이후 세션] 셀 0 (진입점) → 셀 3~7
```

| 셀 | 목적 | 실행 조건 |
|:---:|---|---|
| A | 패키지 설치 | **최초 1회만** → 런타임 재시작 |
| B | 최초 clone | **최초 1회만** |
| 0 | 진입점 (Drive + 경로 + 패키지 확인) | **매 세션마다** |
| 1 | 데이터 갱신 (TTL 크롤링) | 선택적 |
| 2 | ChromaDB 구축 | chroma_db 없을 때만 |
| 3 | 7B 모델 Drive 캐시 확인/저장 | 최초 또는 Drive 삭제 시 |
| 4 | 실행 전 체크리스트 | 선택적 |
| 5 | chatbot.sh (추론 + UI) | **핵심 실행** |
| 6 | chat_output.json 확인 | 검증용 |
| 7 | UI만 단독 실행 | 재실행/데모 |


In [17]:
!pip freeze > requirements.txt

In [18]:
!cat requirements.txt | grep torch

pytorch-lightning==2.4.0
torch==2.5.1+cu124
torchao==0.10.0
torchaudio==2.5.1+cu124
torchcodec==0.11.0+cu128
torchdata==0.11.0
torchmetrics==1.9.0
torchsummary==1.5.1
torchtune==0.6.1
torchvision==0.20.1+cu124


In [ ]:
import sys
import torch

print(sys.version)
print(torch.__version__)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
2.11.0+cu128


In [6]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 A] 패키지 설치 — 최초 1회만 실행
# 완료 후 [런타임] > [런타임 다시 시작] 클릭
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 버전 선택 근거:
#   transformers==4.47.1 : extra_special_tokens list→dict 버그 4.47.0에서 공식 수정
#   tokenizers==0.21.0   : Qwen2.5 tokenizer.json 포맷 호환 (0.19는 파싱 불가)
#   sentence-transformers==3.0.1: 3.4+의 torchcodec 충돌 방지

import subprocess, re, sys

# ── Step 1: CUDA 버전 자동 감지 ──────────────────────────────────
r = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
m = re.search(r"release (\d+)\.(\d+)", r.stdout)
if m:
    major, minor = int(m.group(1)), int(m.group(2))
    cuda_tag = "cu121" if (major == 12 and minor <= 1) else "cu124"
    print(f"CUDA {major}.{minor} 감지 -> wheel: {cuda_tag}")
else:
    cuda_tag = "cu124"
    print(f"CUDA 감지 실패 -> fallback: {cuda_tag}")

# ── Step 2: torch 2.5.1 먼저 고정 설치 ───────────────────────────
print("\n[Step 2] torch==2.5.1 설치 중...")
get_ipython().system(f"{sys.executable} -m pip install -q "
    f"torch==2.5.1+{cuda_tag} "
    f"torchvision==0.20.1+{cuda_tag} "
    f"torchaudio==2.5.1+{cuda_tag} "
    f"--index-url https://download.pytorch.org/whl/{cuda_tag}")

# ── Step 3: 나머지 패키지 설치 ───────────────────────────────────
print("\n[Step 3] 나머지 패키지 설치 중...")
get_ipython().system("""pip install -q     transformers==4.47.1     tokenizers==0.21.0     accelerate==0.34.2     bitsandbytes==0.43.3     datasets==2.21.0     scikit-learn==1.5.1     sentence-transformers==3.0.1     chromadb==0.5.5     langchain==0.2.16     langchain-community==0.2.17     rank_bm25==0.2.2     kiwipiepy==0.17.1     gradio==4.42.0     "pytorch-lightning==2.4.0"     requests==2.32.3     beautifulsoup4==4.12.3     lxml==5.3.0     selenium==4.24.0     webdriver-manager==4.0.2     pdfplumber==0.11.4     tqdm==4.66.5     python-dotenv==1.0.1""")

# ── Step 4: torch 2.5.1 재고정 ───────────────────────────────────
print("\n[Step 4] torch 2.5.1 재고정 중...")
get_ipython().system(f"{sys.executable} -m pip install -q "
    f"torch==2.5.1+{cuda_tag} "
    f"torchvision==0.20.1+{cuda_tag} "
    f"torchaudio==2.5.1+{cuda_tag} "
    f"--index-url https://download.pytorch.org/whl/{cuda_tag}")

# ── 최종 검증 ─────────────────────────────────────────────────────
print()
print("=" * 52)
print("  최종 환경 검증")
print("=" * 52)

for pkg, expected in [
    ("torch",         "2.5.1"),
    ("transformers",  "4.47.1"),
    ("tokenizers",    "0.21.0"),
    ("bitsandbytes",  "0.43.3"),
]:
    r2 = subprocess.run([sys.executable, "-c",
        f"import {pkg}; print({pkg}.__version__)"],
        capture_output=True, text=True)
    ver = r2.stdout.strip()
    ok  = ver.startswith(expected)
    print(f"  {'OK  ' if ok else 'WARN'} {pkg:20s}: {ver}")

pv = sys.version_info
pv_str = f"{pv.major}.{pv.minor}.{pv.micro}"
ok_py = pv.major == 3 and pv.minor == 10
print(f"  {'OK  ' if ok_py else 'WARN'} {'Python':20s}: {pv_str}" +
      (" (과제 요구: 3.10.12 — Colab 고정값)" if not ok_py else ""))

r3 = subprocess.run([sys.executable, "-c",
    "import torch; print(torch.cuda.is_available()); "
    "print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')"],
    capture_output=True, text=True)
cl = r3.stdout.strip().splitlines()
cuda_ok = len(cl) >= 1 and cl[0] == "True"
gpu_name = cl[1] if len(cl) >= 2 else "N/A"
print(f"  {'OK  ' if cuda_ok else 'WARN'} {'CUDA':20s}: {gpu_name}")

print()
print("설치 완료!")
print("→ [런타임] > [런타임 다시 시작] 클릭 후 셀 B 실행")


CUDA 12.8 감지 -> wheel: cu124

[Step 2] torch==2.5.1 설치 중...

[Step 3] 나머지 패키지 설치 중...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 105.3 MB/s eta 0:00:00

[Step 4] torch 2.5.1 재고정 중...

  최종 환경 검증
  OK   torch               : 2.5.1+cu124
  OK   transformers        : 4.47.1
  OK   tokenizers          : 0.21.0
  OK   bitsandbytes        : 0.43.3
  WARN Python              : 3.12.13 (과제 요구: 3.10.12 — Colab 고정값)
  OK   CUDA                : Tesla T4

설치 완료!
→ [런타임] > [런타임 다시 시작] 클릭 후 셀 B 실행


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 B] 레포 Clone — 최초 1회만 실행
# 이미 clone 됐으면 건너뛰고 셀 0 실행
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os

PROJECT = '/content/drive/MyDrive/NLP_Term_Project'

if os.path.exists(PROJECT):
    print(f"이미 존재: {PROJECT}")
    print("→ Clone 건너뜀. 셀 0 실행으로 이동하세요.")
else:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive
    !git clone https://github.com/capyjin/NLP_Term_Project.git
    print("Clone 완료!")
    !ls NLP_Term_Project/

이미 존재: /content/drive/MyDrive/NLP_Term_Project
→ Clone 건너뜀. 셀 0 실행으로 이동하세요.


In [26]:
# 1. chroma_db 삭제
import shutil, os
if os.path.exists('chroma_db'):
    shutil.rmtree('chroma_db')
    print("✓ chroma_db 삭제 완료")

✓ chroma_db 삭제 완료


In [19]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 0] 진입점 — 매 세션마다 가장 먼저 실행
# Drive 마운트 + 경로 설정 + git pull + 패키지 확인
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from google.colab import drive
drive.mount("/content/drive")

import os, sys, importlib
PROJECT = "/content/drive/MyDrive/NLP_Term_Project"
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
print("작업 경로:", os.getcwd())

# Python 버전 확인
py = sys.version_info
py_str = f"{py.major}.{py.minor}.{py.micro}"
if py.major == 3 and py.minor == 10:
    print(f"OK  Python {py_str}")
else:
    print(f"WARN Python {py_str} (과제 요구: 3.10.12)")

# GPU + torch 버전 확인
import torch
torch_ver = torch.__version__
if torch_ver.startswith("2.5.1"):
    print(f"OK  torch {torch_ver}")
else:
    print(f"WARN torch {torch_ver} != 2.5.1 — 셀 A 재실행 후 런타임 재시작 필요")

if torch.cuda.is_available():
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {gb:.1f} GB")
else:
    print("CPU 환경 — [런타임] > [런타임 유형 변경] > T4 GPU 선택 필요")

# 최신 코드 Pull
get_ipython().system("git pull origin main")

# 패키지 확인
print()
for pkg in ["transformers", "bitsandbytes", "kiwipiepy", "rank_bm25", "chromadb", "gradio", "sentence_transformers"]:
    try:
        importlib.import_module(pkg)
        print(f"  OK {pkg}")
    except ImportError:
        print(f"  !! {pkg} 없음 -> 셀 A 실행 후 런타임 재시작 필요")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
작업 경로: /content/drive/MyDrive/NLP_Term_Project
WARN Python 3.12.13 (과제 요구: 3.10.12)
OK  torch 2.5.1+cu124
GPU: Tesla T4 | VRAM: 15.6 GB
From https://github.com/capyjin/NLP_Term_Project
 * branch            main       -> FETCH_HEAD
Updating 3518871..a5ecefe
error: Your local changes to the following files would be overwritten by merge:
	requirements.txt
Please commit your changes or stash them before you merge.
error: The following untracked working tree files would be overwritten by merge:
	data/test_chat.json
	data/test_cls.json
Please move or remove them before you merge.
Aborting

  OK transformers
  OK bitsandbytes
  OK kiwipiepy
  OK rank_bm25
  OK chromadb
  OK gradio
  OK sentence_transformers


In [8]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 1] 데이터 갱신 — 식단/셔틀/공지 크롤링
# TTL: 식단 6h / 셔틀 24h / 공지 6h
# 이미 최신이면 건너뛰어도 됨
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
!python scripts/refresh_data.py
# 특정 항목만 갱신하려면:
# !python scripts/refresh_data.py --meal      # 식단만
# !python scripts/refresh_data.py --shuttle   # 셔틀만
# !python scripts/refresh_data.py --notice    # 공지/장학만


  CNU ChatBot — 데이터 갱신

  식단 데이터 갱신
  → 식단 크롤링 중... (meal_crawler.py)
  ✓ 식단 갱신 완료
  ✓ meal_menu.json: 3건 저장됨

  셔틀버스 데이터 갱신
  → 셔틀버스 크롤링 중... (shuttle_crawler.py)
  ✓ 셔틀버스 갱신 완료

  공지/장학 데이터 갱신
  → 공지/장학 크롤링 중... (cnu_crawler.py)
  ⚠ 공지/장학 크롤링 실패 — 기존 파일 유지
     #17 0x7d728490fac3 <unknown>

  학사일정 데이터 갱신
  ✓ 학사일정 파일 유효 (갱신된 지 15.1시간) — 스킵

  벡터 DB 재구축
  → FAQ 재삽입 중...
  ✓ FAQ 재삽입 완료
  → ChromaDB 임베딩 중... (~5~10분)
  ✓ 벡터 DB 재구축 완료

  완료
  ✓ 데이터 갱신 완료. UI 실행: python src/chatbot_ui.py


In [27]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 2] ChromaDB 구축 — chroma_db 없을 때만 실행
# 있으면 건너뜀 (Drive에 영구 저장됨)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, shutil, json

# FAQ inject 상태 확인
chunks_path = 'data/processed/chunks.json'
if os.path.exists(chunks_path):
    with open(chunks_path, encoding='utf-8') as f:
        chunks = json.load(f)
    faq_count = sum(1 for c in chunks if c.get('source_type') == 'faq_manual')
    print(f"chunks.json: {len(chunks)}개 (FAQ: {faq_count}개)")
    if faq_count < 23:
        print("FAQ 부족 -> inject 실행")
        !python scripts/inject_faq.py
    else:
        print("FAQ OK")

# chroma_db 구축
if os.path.exists('chroma_db') and len(os.listdir('chroma_db')) > 0:
    print("chroma_db 이미 존재 -> 건너뜀")
    print("  (재구축 필요 시: shutil.rmtree('chroma_db') 후 재실행)")
else:
    print("chroma_db 구축 시작...")
    !python src/vectordb/build_db.py
    if os.path.exists('chroma_db'):
        print("chroma_db 구축 완료!")
    else:
        print("구축 실패 - 위 오류 확인 필요")

chunks.json: 205개 (FAQ: 47개)
FAQ OK
chroma_db 구축 시작...
청크 로드: 205건 (크롤링 158건 + FAQ 47건)
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
You try to use a model that was created with version 3.3.1, however, your version is 3.0.1. This might cause unexpected behavior or errors. In that case, try to update to the latest version.



문서 205건 색인 완료
벡터 DB 구축 완료 → /content/drive/MyDrive/NLP_Term_Project/chroma_db
chroma_db 구축 완료!


In [10]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 3] Qwen2.5-3B Drive 캐시 확인/저장
# Drive에 없으면 HuggingFace에서 다운로드 (~3GB, 3~5분)
# 이미 저장됐으면 즉시 완료
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

DRIVE_3B = '/content/drive/MyDrive/models/qwen2.5-3b-4bit'
HF_3B    = 'Qwen/Qwen2.5-3B-Instruct'

if os.path.exists(DRIVE_3B) and os.path.exists(os.path.join(DRIVE_3B, 'config.json')):
    print(f"3B 모델 Drive 캐시 확인: {DRIVE_3B}")
    print("  -> 이미 저장됨. chatbot.sh에서 Drive 경로로 빠르게 로드됩니다.")
else:
    print(f"Drive 캐시 없음 -> {HF_3B} 다운로드 시작 (~3GB, 3~5분)")
    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    model = AutoModelForCausalLM.from_pretrained(
        HF_3B,
        quantization_config=quant,
        device_map={'': 0},        # GPU 0 강제 (CPU offload 방지)
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(HF_3B)
    os.makedirs(DRIVE_3B, exist_ok=True)
    model.save_pretrained(DRIVE_3B)
    tokenizer.save_pretrained(DRIVE_3B)
    print(f"Drive 저장 완료: {DRIVE_3B}")
    del model  # VRAM 해제
    import gc; gc.collect(); torch.cuda.empty_cache()
    print("VRAM 해제 완료. chatbot.sh 실행 준비됨.")

3B 모델 Drive 캐시 확인: /content/drive/MyDrive/models/qwen2.5-3b-4bit
  -> 이미 저장됨. chatbot.sh에서 Drive 경로로 빠르게 로드됩니다.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 4] 실행 전 체크리스트
# 모든 항목이 OK여야 chatbot.sh 정상 실행 가능
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, json

checks = {
    "chroma_db 구축됨": os.path.exists('chroma_db') and len(os.listdir('chroma_db')) > 0,
    "chunks.json 존재": os.path.exists('data/processed/chunks.json'),
    "meal_menu.json 존재": os.path.exists('data/raw/meal_menu.json'),
    "shuttle_bus.json 존재": os.path.exists('data/raw/shuttle_bus.json'),
    "test_chat.json 존재": os.path.exists('data/test_chat.json'),
    "7B Drive 캐시 존재": os.path.exists('/content/drive/MyDrive/models/qwen2.5-7b-4bit/config.json'),
    "chatbot.sh 존재": os.path.exists('chatbot.sh'),
    "src/chatbot_ui.py 존재": os.path.exists('src/chatbot_ui.py'),
}

all_ok = True
for name, ok in checks.items():
    status = 'OK' if ok else '!!'
    print(f"  [{status}] {name}")
    if not ok: all_ok = False

# test_chat.json 없으면 placeholder 생성
if not os.path.exists('data/test_chat.json'):
    placeholder = [
        {"id": 1, "question": "졸업 학점이 몇 점인가요?"},
        {"id": 2, "question": "장학금 신청은 어디서 해?"},
        {"id": 3, "question": "수강신청 변경 기간은 언제야?"},
        {"id": 4, "question": "오늘 학식 뭐 나와?"},
        {"id": 5, "question": "셔틀버스 시간표 알려줘"},
    ]
    with open('data/test_chat.json', 'w', encoding='utf-8') as f:
        json.dump(placeholder, f, ensure_ascii=False, indent=2)
    print("  -> test_chat.json placeholder 생성 완료")

print()
if all_ok:
    print("모든 준비 완료 -> 셀 5 (chatbot.sh) 실행하세요!")
else:
    print("!! 위 항목 확인 후 해당 셀 재실행 필요")

  [OK] chroma_db 구축됨
  [OK] chunks.json 존재
  [OK] meal_menu.json 존재
  [OK] shuttle_bus.json 존재
  [OK] test_chat.json 존재
  [OK] 7B Drive 캐시 존재
  [OK] chatbot.sh 존재
  [OK] src/chatbot_ui.py 존재

모든 준비 완료 -> 셀 5 (chatbot.sh) 실행하세요!


In [28]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 5] chatbot.sh 실행
# 전체 파이프라인: 크롤 -> 추론 -> UI
#   [3/4] test_chat.json -> outputs/chat_output.json 생성
#   [4/4] Gradio UI 실행 -> public URL 출력
# Drive 캐시 있으면 ~30초, 없으면 ~30분 소요
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
!bash chatbot.sh

  CNU Campus ChatBot

[2/4] 식단/셔틀버스 데이터 업데이트 중...
[meal_crawler] 주간 식단 크롤링 시작
  2026-06-08 (월): 6건
  2026-06-09 (화): 6건
  2026-06-10 (수): 6건
  2026-06-11 (목): 6건
  2026-06-12 (금): 5건
  2026-06-13: 메뉴 없음 (운영 없는 날)
[meal_crawler] 저장 완료: /content/drive/MyDrive/NLP_Term_Project/data/raw/meal_menu.json (총 29건)
[shuttle_crawler] 크롤링 시작: https://plus.cnu.ac.kr/html/kr/sub05/sub05_050403.html
  [parser] 크롤링 성공: 2노선
[shuttle_crawler] 저장 완료: /content/drive/MyDrive/NLP_Term_Project/data/raw/shuttle_bus.json (2노선)

[3/4] 챗봇 일괄 추론: data/test_chat.json → outputs/chat_output.json
입력 로드: 5건 (/content/drive/MyDrive/NLP_Term_Project/data/test_chat.json)
ChatBot 초기화 중... (첫 실행 시 모델 다운로드 ~10분)
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
You try to use a model that was created with version 3.3.1, however, your version is 3.0.1. This m

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 6] chat_output.json 결과 확인
# chatbot.sh 완료 후 실행
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import json, os

path = 'outputs/chat_output.json'
if os.path.exists(path):
    with open(path, encoding='utf-8') as f:
        result = json.load(f)
    print(f"chat_output.json 생성 완료 - {len(result)}건\n")
    for i, item in enumerate(result, 1):
        q = item.get('question', item.get('user', ''))
        a = item.get('answer', item.get('model', ''))
        src = item.get('source', '')
        print(f"Q{i}: {q}")
        print(f"A{i}: {a[:120]}..." if len(a) > 120 else f"A{i}: {a}")
        if src:
            print(f"   [경로: {src}]")
        print()
else:
    print("chat_output.json 없음 - chatbot.sh 실행 중 오류 확인")

chat_output.json 생성 완료 - 3건

Q1: 졸업 학점이 몇 점인가요?
A1: 충남대학교 졸업요건은 학과, 입학연도, 전공(단일/복수/부전공) 여부에 따라 달라집니다.

일반적인 졸업 요건 구성:
  • 총 이수학점: 130~140학점 내외 (학과별 상이)
  • 전공 이수: 전공필수 전부 +...

Q2: 장학금 신청은 어디서 해?
A2: 장학금 신청은 다음과 같은 곳에서 진행됩니다:

1. **국가장학금**: 
   - 한국장학재단 홈페이지(www.kosaf.go.kr)에서 신청합니다.
   - 신청 절차는 아래와 같습니다:
     1. 한국장학재...

Q3: 수강신청 변경 기간은 언제야?
A3: 수강신청 및 관련 학사일정은 학기마다 다르게 운영됩니다.

수강신청 주요 안내:
  • 예비 수강신청: 학기 시작 2~3주 전 (희망 과목 선점)
  • 본 수강신청: 학기 시작 1~2주 전 (확정)
  • 수강정정...



In [24]:
!git fetch origin && git reset --hard origin/main

remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 1), reused 3 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 2.58 KiB | 18.00 KiB/s, done.
From https://github.com/capyjin/NLP_Term_Project
 * [new branch]      safe/cnu-chatbot-working-20260609 -> origin/safe/cnu-chatbot-working-20260609
Updating files: 100% (112/112), done.
HEAD is now at a5ecefe fix: 미래 날짜가 최근 공지로 오인되는 버그 수정


In [ ]:
from pathlib import Path
path = Path("src/chatbot_ui.py")
src = path.read_text(encoding="utf-8")

old = '      <h1>🎓 CNU Campus ChatBot</h1>'
new = '      <h1 style="color:#ffffff;">🎓 CNU Campus ChatBot</h1>'

if old in src:
    path.write_text(src.replace(old, new), encoding="utf-8")
    print("✓ 패치 완료 → 셀 7 재실행")
else:
    print("이미 적용됨")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 7] UI 단독 실행 (데모/재실행용)
# chatbot.sh 없이 UI만 띄울 때 사용
# 이미 chatbot.sh로 UI가 실행 중이면 건너뜀
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 옵션 1: 데이터 갱신 + UI
!bash run_ui.sh

# 옵션 2: 데이터 갱신 건너뛰고 UI만
# !bash run_ui.sh --skip

# 옵션 3: UI 직접 실행
# !python src/chatbot_ui.py

  CNU Campus ChatBot — UI 실행

[1/2] 데이터 갱신 중...

  CNU ChatBot — 데이터 갱신

  식단 데이터 갱신
  → 식단 크롤링 중... (meal_crawler.py)
  ✓ 식단 갱신 완료
  ✓ meal_menu.json: 3건 저장됨

  셔틀버스 데이터 갱신
  → 셔틀버스 크롤링 중... (shuttle_crawler.py)
  ✓ 셔틀버스 갱신 완료

  공지/장학 데이터 갱신
  → 공지/장학 크롤링 중... (cnu_crawler.py)
  ⚠ 공지/장학 크롤링 실패 — 기존 파일 유지
     #17 0x7f52a593aac3 <unknown>

  학사일정 데이터 갱신
  ✓ 학사일정 파일 유효 (갱신된 지 0.2시간) — 스킵

  벡터 DB 재구축
  → FAQ 재삽입 중...
  ✓ FAQ 재삽입 완료
  → ChromaDB 임베딩 중... (~5~10분)
  ✓ 벡터 DB 재구축 완료

  완료
  ✓ 데이터 갱신 완료. UI 실행: python src/chatbot_ui.py

[2/2] UI 실행 중...
  접속: http://localhost:7860  (Colab: share URL 확인)
  종료: Ctrl+C

  충남대학교 Campus ChatBot 로딩 중...
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
You try to use a model that was created with version 3.3.1, however, your version is 3.0.1. This might cause unexpected behavior o